# Chapter 04 — APIs, Web Data, and Schema Validation

External systems expose data through HTTP APIs or web pages. Learn HTTP request structure, JSON response handling, defensive parsing, HTML extraction, and schema validation.

## Prerequisites
Dictionaries, JSON, exceptions, and basic DataFrames; live internet access is optional.

## Learning goals
- Explain URLs, methods, status codes, headers, query parameters, and timeouts.
- Use Requests to retrieve and check API responses.
- Parse HTML locally with BeautifulSoup.
- Validate records with Pydantic and report malformed data.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Guided learning path

Work from top to bottom. Each step states the purpose, explains the API choice, and provides a complete code cell. Modify parameters and predict the output before running it.

### Step 1 — Guided 1 — Inspect the response envelope

Many APIs wrap records in metadata such as status, count, pagination tokens, or errors. Validate the envelope before trusting a nested list.

In [2]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import json
payload=json.loads((DATA/"sample_api_response.json").read_text(encoding="utf-8"))
assert payload.get("status")=="ok"
assert payload.get("count")==len(payload.get("results",[]))
print("Envelope valid; records:",len(payload["results"]))

Data directory: /mnt/data/python_data_analysis_book/data
Envelope valid; records: 3


### Step 2 — Guided 2 — Parse the HTML catalogue

CSS `.product` selects class product; `li.product[data-id]` additionally requires an `li` and a data-id attribute.

In [3]:
from bs4 import BeautifulSoup
html=(DATA/"sample_page.html").read_text(encoding="utf-8")
soup=BeautifulSoup(html,"html.parser")
parsed=[{"id":int(x["data-id"]),"text":x.get_text(" ",strip=True)} for x in soup.select("li.product[data-id]")]
print(parsed)

[{'id': 1, 'text': 'Notebook - 120'}, {'id': 2, 'text': 'Pen - 15'}, {'id': 3, 'text': 'Backpack - 850'}]


### Step 3 — Guided 3 — Validate with Pydantic

Type annotations and Field constraints make the contract explicit. Keep valid and invalid records separate so malformed data can be reviewed.

In [4]:
from pydantic import BaseModel, Field, ValidationError
class Product(BaseModel):
 id:int; name:str=Field(min_length=1); price:float=Field(gt=0)
valid=[]; invalid=[]
for item in payload["results"]:
 try: valid.append(Product.parse_obj(item).dict())
 except ValidationError as exc: invalid.append(str(exc))
print("valid:",len(valid),"invalid:",len(invalid))

valid: 3 invalid: 0


/mnt/data/python_data_analysis_book/Chapter_04_apis_web_data_and_schema_validation/Chapter_04_Step_by_Step_Tutorial.ipynb:6: PydanticDeprecatedSince20:

The `parse_obj` method is deprecated; use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/

/mnt/data/python_data_analysis_book/Chapter_04_apis_web_data_and_schema_validation/Chapter_04_Step_by_Step_Tutorial.ipynb:6: PydanticDeprecatedSince20:

The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/



### Step 4 — Guided 4 — Build a defensive HTTP function

A function definition can be tested locally by inspecting its signature; live invocation is deliberately kept separate.

In [5]:
import inspect, requests
def get_records(url, params=None, timeout=(5,20)):
    response=requests.get(url,params=params,timeout=timeout)
    response.raise_for_status()
    data=response.json()
    if not isinstance(data,dict) or "results" not in data: raise ValueError("Unexpected API response shape")
    return data["results"]
print(inspect.signature(get_records))

(url, params=None, timeout=(5, 20))


## Tutorial checkpoint

1. What was the input data type?
2. What did the function or method return?
3. Which argument most changed the result?
4. What edge case should be tested before using this in production?